# PyMIF beginner notebook 07 - create a new Zarr dataset with labels inside

This notebook starts from two arrays:

- one intensity image array,
- one label array.

It writes a single OME-Zarr where the image is at the root and the labels are inside `/labels/nuclei`.

In [ ]:
from pathlib import Path
import shutil
import sys
import numpy as np

# Use the installed package when available. When running from a local PyMIF
# source checkout, this fallback adds the repository root to sys.path.
try:
    import pymif.microscope_manager as mm
except ModuleNotFoundError:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "pymif").is_dir():
            sys.path.insert(0, str(candidate))
            break
    import pymif.microscope_manager as mm

OUTPUT_DIR = Path("pymif_tutorial_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("PyMIF managers loaded from:", mm.__file__)
print("Tutorial output folder:", OUTPUT_DIR.resolve())

In [ ]:
def summarize_manager(manager, title="dataset"):
    """Print the fields beginners usually need to inspect first."""
    print(f"\n{title}")
    print("-" * len(title))
    print("manager:", type(manager).__name__)
    print("axes:", manager.metadata.get("axes"))
    print("data_type:", manager.metadata.get("data_type", "intensity"))
    print("levels:", len(manager.data))
    for i, arr in enumerate(manager.data):
        print(f"  level {i}: shape={arr.shape}, chunks={getattr(arr, 'chunksize', None)}, dtype={arr.dtype}")
    for key in ["scales", "units", "time_increment", "time_increment_unit", "channel_names", "channel_colors"]:
        print(f"{key}:", manager.metadata.get(key))


def clean_path(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    return path

In [ ]:
def label_metadata_from_image_metadata(image_metadata, name="nuclei", dtype="uint32"):
    """Make safe label metadata from image metadata by removing the channel axis.

    Most segmentation labels are one integer label image per time point and do
    not have an image channel axis. This helper avoids confusion from legacy
    TCZYX examples by creating explicit TZYX label metadata.
    """
    md = dict(image_metadata)
    axes = str(md["axes"]).lower()
    if "c" in axes:
        c_axis = axes.index("c")
        md["axes"] = "".join(ax for ax in axes if ax != "c")
        md["size"] = [tuple(v for i, v in enumerate(size) if i != c_axis) for size in md["size"]]
        md["chunksize"] = [tuple(v for i, v in enumerate(chunks) if i != c_axis) for chunks in md["chunksize"]]
    md["name"] = name
    md["dtype"] = dtype
    md["data_type"] = "label"
    md["is_label"] = True
    md["channel_names"] = []
    md["channel_colors"] = []
    return md

## Create example arrays

In [ ]:
rng = np.random.default_rng(30)
raw = rng.integers(0, 3000, size=(1, 2, 6, 80, 80), dtype=np.uint16)  # T, C, Z, Y, X
labels = np.zeros((1, 6, 80, 80), dtype=np.uint32)                  # T, Z, Y, X
labels[:, :, 12:35, 15:40] = 1
labels[:, :, 45:70, 35:65] = 2

raw_metadata = {
    "name": "new_dataset_raw",
    "axes": "tczyx",
    "size": [raw.shape],
    "chunksize": [(1, 1, 3, 40, 40)],
    "scales": [(1.0, 0.20, 0.20)],
    "units": ("micrometer", "micrometer", "micrometer"),
    "time_increment": 1.0,
    "time_increment_unit": "second",
    "channel_names": ["DAPI", "GFP"],
    "channel_colors": ["0000FF", "00FF00"],
    "dtype": "uint16",
    "data_type": "intensity",
}

raw_manager = mm.ArrayManager(raw, raw_metadata, chunks=raw_metadata["chunksize"][0])
raw_manager.build_pyramid(num_levels=3, downscale_factor=(1, 2, 2))
summarize_manager(raw_manager, "raw manager")

## Step 1: write the raw image to the root

In [ ]:
out = clean_path(OUTPUT_DIR / "new_dataset_with_labels.zarr")
raw_manager.to_zarr(out, ngff_version="0.5", zarr_format=3)
print("Wrote raw image root:", out)

## Step 2: reopen in write mode and create the label group

Use `mode="r+"` for modifying an existing store.

In [ ]:
z = mm.ZarrManager(out, mode="r+")
label_md = label_metadata_from_image_metadata(z.metadata, name="nuclei", dtype="uint32")
z.create_empty_group("nuclei", label_md, is_label=True)
print("Created label group. Labels now:", list(z.labels.keys()))

## Step 3: write labels into `/labels/nuclei`

In [ ]:
z.write_label_region(labels, group="labels/nuclei", level=0, downscale_factor=(1, 2, 2))
z.read()
print("Labels after writing:", list(z.labels.keys()))
print(z.labels["nuclei"])

## Step 4: verify root image and labels together

In [ ]:
read_back = mm.ZarrManager(out, mode="r")
print("root levels:", len(read_back.raw.data))
print("label levels:", len(read_back.labels["nuclei"].data))
print("root shape:", read_back.raw.data[0].shape)
print("label shape:", read_back.labels["nuclei"].data[0].shape)
print("label unique values:", np.unique(read_back.labels["nuclei"].data[0].compute()))

## Optional: add another image group before saving a final copy

In [ ]:
z = mm.ZarrManager(out, mode="r+")
corrected_md = dict(z.metadata)
corrected_md["name"] = "background_corrected"
z.create_empty_group("background_corrected", corrected_md, is_label=False)
corrected = np.maximum(raw.astype(np.int32) - 100, 0).astype(np.uint16)
z.write_image_region(corrected, group="background_corrected", level=0, downscale_factor=(1, 2, 2))
z.read()

final_out = clean_path(OUTPUT_DIR / "new_dataset_with_labels_and_group_copy.zarr")
z.to_zarr(final_out, include_groups=True, include_labels=True, ngff_version="0.5", zarr_format=3)
final = mm.ZarrManager(final_out, mode="r")
print("Final image groups:", list(final.groups.keys()))
print("Final label groups:", list(final.labels.keys()))